### Prompt Engineering

#### Basic prompting

- Methods:
    - Zero shot
    - Few shot
    - Role prompting (with contenxt)
    - Persona prompting (with contenxt)

__Zero shot__

Translate:
Hello


__Few shot__

- Example 1 (Translation)

    - Hello, how are you doing?: سلام، چی کار میکنی؟
    - What's up?: 

- Example 2 (Classification)

    - I'm so happy //Positive
    - I'm worried. //Negative
    - This is bad  //


__Role Prompting__

- Important note: role without context doesn't work!
- Important note: don't ask simple question like "how does generation work?" ask details: "what's the difference between .. and ..?"
- __Context__ means: past conversation, background info, your situation, limitation, and details question

- __Persona prompting__:

    - role: You're an AI engineer
    - Persona: You're an AI engineer who explains every concept in simple terms with coding example


- __Example 1__: No context 

    - You're an expert AI engineering tell me about RAG paradigm

- __Example 2__: with context
    - __Role(Persona)__:
    - You are a senior AI engineer who deploys RAG systems in production using LangChain and LangGraph. 
    - You have solved latency and cost issues for enterprise clients.

    - __Context (My Situation)__:

    - I am a mid-level Python developer. I have already built the retrieval part (I use ChromaDB and get 5 relevant chunks per query).

    - My data is PDF product manuals (each chunk is ~1,500 tokens, so 5 chunks = ~7,500 tokens).

    - My input prompts are short (under 50 tokens).

    - __Critical constraint__: 
    - My system serves real-time web users, so latency must be under 1.5 seconds. I cannot afford to call the LLM more than once per request.

    - __My Specific Questions__:

    - In LangChain, what is the difference between "stuff", "map_reduce", and "refine" chains?
    - Which one should I use given my 5 chunks (~8,000 tokens) and my strict latency limit?
    - If I use "stuff", will I hit the token limit of GPT-3.5-Turbo (16k context)? Show me how to calculate it.
    - Give me a code snippet for the fastest possible method, and tell me what trade-off I am making (cost vs. accuracy).


### Reasoning Prompting

- Methods:

    - CoT
    - Self Consistency
    - ToT
    - GoT

- __CoT__

    - Think step by step
    - Zero shot COT, Few shot COT

- __Self Consistency__

    - one prompt, multiple answers, voting. 
    - usually use high temperature (increase the creativity)

- __ToT__

    - Different ideas, Evaluate, Expand and Prune درخت جستجو با قابلیت تعمیم شاخه های مفید
    - Not only return one answer, produce multiple ideas

- __Graph of Thougts__

    - Not a tree, it's a graph
    - Nodes can connect together again. امکان بازگشت به عقب و بازنگری دارد

### Decomposition Prompting

- __Sequential Decomposition__

    - break prompt into sequential steps

- __Parallel Decomposition__

    - break prompt into Parallel steps

- __Hierarchical Decomposition__

    - break prompt into Hierarchical steps

    - Example:

        - Design a project management app. Break it down hierarchically:

            Level 1 (Main Modules):
            1. User Management
            2. Project Management
            3. Reporting

            Level 2 (Sub-modules):
            
                1.1. User registration and login
                1.2. Authentication and roles
                2.1. Create/delete projects
                2.2. Task assignment
                2.3. Progress tracking
                3.1. Daily reports
                3.2. Weekly reports

            For each sub-module, describe its features and APIs.

### Parameter-Efficient Learning 

#### Soft Prompt Methods

- __Prompt Tuning__
    - The entire base model is frozen — no weights are updated during training.

    - The input text is tokenized and mapped to embedding vectors as usual.

    - A small number of trainable vectors (typically 1 to 32) — referred to as virtual tokens — are prepended to the input embedding sequence. These vectors are randomly initialized at the start of training.

    - A small dataset (a few hundred examples) is required to train these virtual tokens.

    - During backpropagation, only these prepended vectors are updated; all other model parameters remain untouched.

    - During inference, the presence of these learned vectors steers the model's 
    behavior toward a specific style, tone, or domain — without needing repetitive hard prompts like "You are a helpful assistant."

    
- __Prefix Tuning__

    - Similar to Prompt Tuning, 
    
    - but instead of only modifying the input layer, 
    
    - trainable vectors are added to the Key (K) and Value (V) representations in every attention layer of the model.

    - The Query (Q) remains unchanged.

    - However, it requires more memory and computation compared to standard Prompt Tuning.

    - یعنی بردارهای قابل اموزش نه تنها به ابتدای بردار های امبدینگ بلکه به بردار های کلید و مقدار ماتریس های توجه هم اضافه میشن

    - بردار کویری تغییری نمی کند

    - بردار کویری عوض نمیشه چون تغییر کنه یعنی ورودی کاربر تغییر میکنه

    -  بردار کلید افزوده میشه انگار داریم برچسب های بیشتری میزنیم تا جستجو بهتر بشه

    - بردار مقدار تغییر میکنه انگار داریم محتوای بیشتری اضافه میکنیم

- Weakness

    - dependens on the length of vectors and vitual tokens

    - Not as good as LoRA on complicated tasks and datasets

    - Complex method in coding and Memory 

#### Adaptors methods


- __LoRA__

    - Mechanism: Two low-rank matrices (A and B) are added to linear layers (such as Q, K, V in Attention, and Feed-Forward layers)

    - Training: The original model weights are completely frozen, and only the A and B matrices are updated during training.

    - Precision: The base model can be loaded in FP16, BF16, or FP32. The LoRA matrices are typically updated in the same precision as the base model.

    - Rank: Usually fixed across all layers (e.g., r=8 or r=16).

    - وزن های مدل اصلی همانگونه که هستند لود میشوند که میتونن ۱۶ بیت یا ۳۲ بیتی باشند
    - مقدار رنک به سایز دیتاست، اینکه مدل از نوع ریزن هستند و پیچیدگی محاسباتی بستگی دارد
    - برای مثال برای دیتاست کوچک مقدار ۸ یا ۱۶ تا مدل اورفیت نشه


- __QLoRA__

    - Mechanism: Combines LoRA + 4-bit quantization of the base model.

    - Number Format: Uses NF4 (Normal Float 4), which is specifically designed for the normal distribution of Transformer weights, preserving better accuracy than integer 4-bit formats.

    - Double Quantization: Quantizes the quantization constants themselves to save even more memory

    - Paged Optimizers: Uses paged memory management on the GPU to prevent Out-of-Memory (OOM) errors


- __AdaLoRA__

    - Mechanism: Similar to LoRA, but with adaptive and dynamic ranks for each layer.

    - Process: During training, ranks are automatically increased or decreased based on the __importance__ of each layer and each matrix.

    - Rank Allocation: More important layers (e.g., deeper or later layers) receive higher ranks, 
    
    - while less important layers (e.g., shallow layers) receive lower ranks.

    - Advantage: Provides better efficiency and more optimized memory usage compared to standard LoRA, especially for very large models.
